In [ ]:
import pm4py

dataPath = "data/PhoenixBLN-NYT_1980-2018.csv"

In [ ]:
from pm4pyExtension.preprocessing.Pheonix import code2text_Pheonix, extract_time_range, import_csv, preprocess

In [ ]:
from pm4pyExtension.algo.discovery import get_action_reaction_graph as discover_reactions
from pm4pyExtension.algo.discovery import get_eventually_follows_in_days_graph
from pm4pyExtension.algo.merging import merge_action_reaction_graphs
from pm4pyExtension.algo.normalize import source_normalize_dfg, target_normalize_dfg
from pm4pyExtension.algo.reduction import max_path_denoise_dfg, minthreshold_dfg


def scale_dfg(dfg):
    #normalize the edges to sum to 1
    absolut_weight = 0
    for weight in dfg.values():
        absolut_weight += weight
    # normalize the weight of transitions into each target node
    for key, weight in dfg.items():
        dfg[key] = weight / absolut_weight

    return dfg


def get_action_reaction_graph(unprocessed_event_log, daysFollow, source_country, target_country,
                              start_date=None, end_date=None):
    actions = preprocess(unprocessed_event_log, key=source_country + target_country,
                         start_date=start_date, end_date=end_date)
    reactions = preprocess(unprocessed_event_log, key=target_country + source_country,
                           start_date=start_date, end_date=end_date)
    return discover_reactions(actions, reactions, daysFollow)

In [ ]:
from pm4pyExtension.tools import (
    make_bpmn, make_dfg, make_efig as make_efg, make_petrinet, make_process_tree, normalize_dfg_to
)

In [ ]:
unprocessed_event_log = import_csv(dataPath)
preprocessed_event_log = preprocess(unprocessed_event_log, key="PSEISR")
case_split_log = pm4py.filter_between(preprocessed_event_log, "Disapprove", "Fight")
dfg = make_dfg(case_split_log, show=True, minThresh=0.2,max_path_denoise=True,show_start_end=True, normalize=False, normalize_to="source")

In [ ]:
#event_log = preprocess(unprocessed_event_log,key="USADEU", start_date="1960-01-01", end_date="2000-01-01")
#example dfg call
#dfg = make_dfg(event_log,
#            max_path_denoise=False, minThresh=0.2, show=True, 
#            normalize=True, normalize_to="source")
#example efg call
#efg = make_efg(event_log,
#            max_path_denoise=False, minThresh=0.2, show=True, 
#            normalize=True, normalize_to="source", daysFollow=3)
#example manual dfg construction
#dfg, start_activities, end_activities = pm4py.discover_dfg(event_log) #for efg; efg = get_eventually_follows_in_days_graph(event_log, daysFollow)
#dfg = normalize_dfg_to(dfg, normalize_to="source")
#dfg = minthreshold_dfg(dfg, 0.2)
#pm4py.view_dfg(dfg, start_activities=None, end_activities=None, format='png')


#get actopm reaction graph
#efg_ar = get_action_reaction_graph(unprocessed_event_log, daysFollow=20, source_country="ISR", target_country="PSE")
#efg_ar = normalize_dfg_to(efg_ar, normalize_to="source")
#efg_ar = minthreshold_dfg(efg_ar, 0.)
#pm4py.view_dfg(efg_ar, start_activities=None, end_activities=None, format='png')

# Compare consecutive legs of Germany → United Kingdom → United States → Germany.
route = ["DEU", "GBR", "USA"]
legs = [
    preprocess(unprocessed_event_log, key=country + route[(i + 1) % len(route)],
               start_date="1980-01-01", end_date="1990-01-01")
    for i, country in enumerate(route)
]
reactions = [
    discover_reactions(leg, legs[(i + 1) % len(legs)], daysFollow=15)
    for i, leg in enumerate(legs)
]

In [ ]:
normalized = [normalize_dfg_to(graph, normalize_to="source") for graph in reactions]
merged = merge_action_reaction_graphs(normalized, ["Germany", "United Kingdom", "United States"])
dfg = minthreshold_dfg(max_path_denoise_dfg(merged), 0.1)
if dfg:
    pm4py.view_dfg(dfg, start_activities=None, end_activities=None, format="png")